# Tratamento silver — Estações

Limpa o inventário de estações (`codigos_estacoes.csv`): remove duplicatas
exatas (o inventário da ANA às vezes lista a mesma estação duas vezes) e
garante o tipo certo de cada coluna.

Entrada (bronze): `data/bronze/ana/estacoes/codigos_estacoes.csv`
Saída (silver):   `data/silver/ana/estacoes/estacoes.csv`

Rode as células em ordem, de cima pra baixo.

In [ ]:
from pathlib import Path

import pandas as pd


def encontrar_raiz_do_projeto(marcador: str = "pyproject.toml") -> Path:
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / marcador).exists():
            return pasta
    raise FileNotFoundError(
        f"Não encontrei '{marcador}' subindo a partir de {atual}. "
        "Este notebook está dentro da pasta do projeto?"
    )


RAIZ = encontrar_raiz_do_projeto()

ENTRADA_CSV = RAIZ / "data" / "bronze" / "ana" / "estacoes" / "codigos_estacoes.csv"
SAIDA_CSV = RAIZ / "data" / "silver" / "ana" / "estacoes" / "estacoes.csv"

print("Raiz do projeto:", RAIZ)
print("Lendo de:", ENTRADA_CSV)
print("Vai salvar em:", SAIDA_CSV)

## 1. Ler o CSV bruto

In [ ]:
bruto = pd.read_csv(ENTRADA_CSV)
print(f"{len(bruto)} linha(s) bruta(s) lida(s).")
print(f"Duplicatas exatas encontradas: {bruto.duplicated().sum()}")
bruto.head()

## 2. Limpar: remover duplicatas e ajustar tipos

In [ ]:
silver = bruto.drop_duplicates().copy()

silver["codigo"] = pd.to_numeric(silver["codigo"], errors="coerce").astype("Int64")
silver["latitude"] = pd.to_numeric(silver["latitude"], errors="coerce")
silver["longitude"] = pd.to_numeric(silver["longitude"], errors="coerce")
silver["tipo"] = silver["tipo"].str.strip()
silver["nome"] = silver["nome"].str.strip()

silver = silver.sort_values(["tipo", "codigo"]).reset_index(drop=True)

print(f"{len(silver)} linha(s) após remover duplicatas (eram {len(bruto)}).")
silver.head()

## 3. Conferências rápidas (sanity checks)

In [ ]:
print("Duplicatas de código restantes (esperado 0, a menos que o mesmo código sirva pra dois tipos):")
print(silver[silver.duplicated(subset=['codigo'], keep=False)].sort_values('codigo'))
print()
print("Nulos por coluna:")
print(silver.isna().sum())
print()
print("Latitude fora do Brasil (aprox. -35 a 6):", ((silver['latitude'] > 6) | (silver['latitude'] < -35)).sum())
print("Longitude fora do Brasil (aprox. -75 a -33):", ((silver['longitude'] > -33) | (silver['longitude'] < -75)).sum())
print()
print("Estações por tipo:")
print(silver["tipo"].value_counts())

## 4. Salvar em silver

In [ ]:
SAIDA_CSV.parent.mkdir(parents=True, exist_ok=True)
silver.to_csv(SAIDA_CSV, index=False, encoding="utf-8")
print(f"Salvo em {SAIDA_CSV}")

## Próximos passos

- Rodar `gerar_parquet_estacoes.ipynb` (gold) em cima desse CSV.
- Essa tabela serve pra juntar (join) com chuva/cota por `codigoestacao` no
  BI, trazendo nome/coordenadas de cada estação pro relatório.